There is a bug in `pytorch` and you need to make sure that in the conda environment the following variable is set:
```
conda env config vars set KMP_DUPLICATE_LIB_OK=TRUE
```
You can check that the variable is set by running:
```
conda env config vars list
```

In [1]:
from collections import defaultdict
import datetime

import numpy as np

import torch
import torchrl
import tensordict

from envs.mh5robotenv import MH5RobotEnv

from tensordict.nn import  TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor

from torch import nn

from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import GymEnv, TransformedEnv, Compose, ObservationNorm, DoubleToFloat, StepCounter
from torchrl.envs.utils import check_env_specs, set_exploration_type, ExplorationType

from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE

from torchrl.record import TensorboardLogger, VideoRecorder

from torchinfo import summary

from tqdm import tqdm


In [2]:
print(f"torch version: {torch.__version__}")
print(f"torchrl version: {torchrl.__version__}")
print(f"tensordict version: {tensordict.__version__}")

torch version: 2.7.1
torchrl version: 0.0.0+unknown
tensordict version: 0.10.0


In [3]:
config = {
    'num_cells': 256,
    'frames_per_batch': 1_000,
    'total_frames': 100_000,
    'gamma': 0.99,
    'lmbda': 0.95,
    'clip_epsilon': 0.2,
    'entropy_eps': 1e-4,
    'lr': 3e-4,
    'num_epochs': 10,
    'sub_batch_size': 64,
    'max_grad_norm': 1.0,
    'timestep': 1.0 / 100.0,
}

In [4]:
device = torch.device("cpu")
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
# if torch.cuda.is_available():
#     device = torch.device("cuda")
print(f"device: {device}")

device: cpu


In [5]:
train_env = TransformedEnv(
    GymEnv("MH5Robot-v8", device=device),
    Compose(
        ObservationNorm(in_keys=['observation']),
        DoubleToFloat(),
        StepCounter(),
    ),
)

In [6]:
train_env.transform[0].init_stats(num_iter=1000, reduce_dim=0, cat_dim=0)
print("normalization constant shape:", train_env.transform[0].loc.shape)

normalization constant shape: torch.Size([59])


In [7]:
print(f"observation_spec:\n{train_env.observation_spec}\n")
print(f"reward_spec:\n{train_env.reward_spec}\n")
print(f"input_spec:\n{train_env.input_spec}\n")
print(f"action_spec:{train_env.action_spec}\n")

observation_spec:
Composite(
    observation: UnboundedContinuous(
        shape=torch.Size([59]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([59]), device=cpu, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([59]), device=cpu, dtype=torch.float32, contiguous=True)),
        device=cpu,
        dtype=torch.float32,
        domain=continuous),
    step_count: BoundedDiscrete(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, contiguous=True),
            high=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, contiguous=True)),
        device=cpu,
        dtype=torch.int64,
        domain=discrete),
    device=cpu,
    shape=torch.Size([]),
    data_cls=None)

reward_spec:
UnboundedContinuous(
    shape=torch.Size([1]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, contiguous=True)

In [8]:
check_env_specs(train_env)

2025-10-26 10:16:43,065 [torchrl][INFO]    check_env_specs succeeded! [END]


In [18]:
rollout = train_env.rollout(3)
print("rollout of three steps:", rollout)
print("Shape of the rollout TensorDict:", rollout.batch_size)

rollout of three steps: TensorDict(
    fields={
        action: Tensor(shape=torch.Size([3, 24]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([3, 59]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                step_count: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.int64, is_shared=False),
                terminated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([3]),
            dev

In [47]:
actor_net = nn.Sequential(
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(2 * train_env.action_spec.shape[-1], device=device),
    NormalParamExtractor(),
)

policy_module = TensorDictModule(actor_net, in_keys=['observation'], out_keys=['loc', 'scale'])

policy_module = ProbabilisticActor(
    module=policy_module,
    in_keys=['loc', 'scale'],
    spec=train_env.action_spec,
    distribution_class=TanhNormal,
    distribution_kwargs={
        'low': train_env.action_spec_unbatched.space.low,
        'high': train_env.action_spec_unbatched.space.high,
    },
    return_log_prob=True,
)

In [48]:
value_net = nn.Sequential(
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(1, device=device)
)

value_module = ValueOperator(
    module=value_net,
    in_keys=['observation'],
)

In [12]:
collector = SyncDataCollector(
    create_env_fn=train_env,
    policy=policy_module,
    frames_per_batch=config['frames_per_batch'],
    total_frames=config['total_frames'],
    split_trajs=False,
    device=device,
    # use_buffers=False,  # https://github.com/pytorch/rl/issues/3066#issuecomment-3077398138
)

In [13]:
replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=config['frames_per_batch']),
    sampler=SamplerWithoutReplacement(),
)

In [14]:
advantage_module = GAE(
    gamma=config['gamma'],
    lmbda=config['lmbda'],
    value_network=value_module,
    average_gae=True,
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=config['clip_epsilon'],
    entropy_bonus=bool(config['entropy_eps']),
    entropy_coeff=config['entropy_eps'],
    critic_coeff=1.0,
    loss_critic_type='smooth_l1',
)

optim = torch.optim.Adam(loss_module.parameters(), config['lr'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optim,
    T_max=(config['total_frames'] // config['frames_per_batch']),
    eta_min=0.0
)

In [ ]:
# we need to do this to "initialize" the Lazy modules
# print("Running policy:", policy_module(rollout))
# print("Running value:", value_module(rollout))


KeyError: "Couldn't find the log-prob action_log_prob in the input data."

In [ ]:
print(summary(policy_module, depth=4))
print(summary(value_module))
print(summary(advantage_module))

In [ ]:
logger = TensorboardLogger(
    exp_name=f"{datetime.datetime.now():%Y%m%d%H%M%S}",
    log_dir="logs")
pbar = tqdm(total=config['total_frames'])
best_eval_sum = 0

for i, tensordict_data in enumerate(collector):
    for _ in range (config['num_epochs']):
        advantage_module(tensordict_data)
        data_view = tensordict_data.reshape(-1)
        replay_buffer.extend(data_view.cpu())
        for _ in range(config['frames_per_batch'] // config['sub_batch_size']):
            subdata = replay_buffer.sample(config['sub_batch_size'])
            loss_vals = loss_module(subdata.to(device))
            loss_value = (
                loss_vals['loss_objective']
                + loss_vals['loss_critic']
                + loss_vals['loss_entropy']
            )
            loss_value.backward()
            torch.nn.utils.clip_grad_norm_(loss_module.parameters(), config['max_grad_norm'])
            optim.step()
            optim.zero_grad()

    pbar.update(tensordict_data.numel())

    logger.log_scalar(
        name="train/reward_mean",
        value=tensordict_data['next', 'reward'].mean().item(),
        step=pbar.n)
    logger.log_scalar(
        name="train/step_count",
        value=tensordict_data['step_count'].max().item(),
        step=pbar.n)
    logger.log_scalar(
        name="train/lr",
        value=optim.param_groups[0]['lr'],
        step=pbar.n)

    if (i + 1) % 10 == 0:
        with set_exploration_type(ExplorationType.DETERMINISTIC), torch.no_grad():
            eval_env = TransformedEnv(
                GymEnv("MH5Robot-v8", device=device, from_pixels=True, pixels_only=False),
                train_env.transform.clone())
            eval_env.append_transform(VideoRecorder(logger, f"eval/{pbar.n}"))
            eval_rollout = eval_env.rollout(1000, policy_module)
            logger.log_scalar(
                name="eval/reward_mean",
                value=eval_rollout['next', 'reward'].mean().item(),
                step=pbar.n)
            logger.log_scalar(
                name="eval/reward_sum",
                value=eval_rollout['next', 'reward'].sum().item(),
                step=pbar.n)
            logger.log_scalar(
                name="eval/step_count",
                value=eval_rollout['step_count'].max().item(),
                step=pbar.n)

            if eval_rollout['step_count'].sum().item() > best_eval_sum:
                best_eval_sum = eval_rollout['step_count'].sum().item()
                eval_env.transform.dump()

            del eval_rollout

    scheduler.step()


In [ ]:
# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 10))
# plt.subplot(2, 2, 1)
# plt.plot(logs["reward"])
# plt.title("training rewards (average)")
# plt.subplot(2, 2, 2)
# plt.plot(logs["step_count"])
# plt.title("Max step count (training)")
# plt.subplot(2, 2, 3)
# plt.plot(logs["eval reward (sum)"])
# plt.title("Return (test)")
# plt.subplot(2, 2, 4)
# plt.plot(logs["eval step_count"])
# plt.title("Max step count (test)")
# plt.show()

In [ ]:
# from torchrl.record import VideoRecorder
# # from torchrl.record.loggers.csv import CSVLogger

# # logger = CSVLogger(exp_name="first training", log_dir="logging", video_format="mp4")
# recorder = VideoRecorder(logger=logger, tag="validation")
# valid_env = TransformedEnv(
#     GymEnv("MH5Robot-v8", from_pixels=True, pixels_only=False),
#     recorder)
# # valid_env.transform[0].init_stats(num_iter=1000, reduce_dim=0, cat_dim=0)
# with set_exploration_type(ExplorationType.DETERMINISTIC), torch.no_grad():
#     valid_env.rollout(1000, policy_module)
# recorder.dump()

In [ ]:
e = MH5RobotEnv()

In [ ]:
e.reset()

In [ ]:
e.step([0.0]*24)

In [ ]:
command = np.array([0.0]*24)

In [ ]:
command[0]=np.nan

In [ ]:
command

In [ ]:
e.step(command)

In [ ]:
import mujoco
dir(mujoco.mjtWarning)

In [ ]:
mujoco.mjtWarning.mjWARN_VGEOMFULL

In [ ]:
e.data.warning[mujoco.mjtWarning.mjNWARNING]

In [ ]:
e1 = GymEnv("MH5Robot-v8")

In [ ]:
e1.output_spec

In [ ]:
e1._env

In [ ]:
e1.spec

In [ ]:
e2 = GymEnv("MH5Robot-v8", reset_noise_scale=2e-4)

In [ ]:
e2.spec

In [ ]:
e3 = GymEnv("MH5Robot-v8", reset_noise_scale=2e-4, max_episode_steps=2000)

In [ ]:
e3.spec

In [ ]:
e.dt

In [ ]:
e.model.opt

In [ ]:
e.dt

In [ ]:
e.render_fps

In [ ]:
e.metadata

In [ ]:
e1.metadata